# 03 · Downstream Pipeline Integration

gwmock writes **signal** and **noise** as separate frames. A search or
inference pipeline wants the **combined strain** `data = signal + noise`
in a standard channel. This notebook shows the hand-off.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260622-et-cosmology-div-online-workshop-sgwb/blob/main/materials/notebooks/03-downstream-integration.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in correlated-noise support used in notebook 04.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 1. A small signal+noise run

(Same offline BBH config as notebook 01.)

In [ ]:
%%writefile population.csv
detector_frame_mass_1,detector_frame_mass_2,coa_time,distance,declination,right_ascension,polarization_angle,inclination
30.0,25.0,1577491300.0,400.0,0.3,1.1,0.2,0.5

In [ ]:
%%writefile config.yaml
globals:
    simulator-arguments:
        sampling-frequency: 2048
        duration: 128
        total-duration: 128
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output
    metadata-directory: metadata
orchestration:
    population:
        backend: FilePopulationLoader
        source-type: bbh
        arguments: {path: population.csv}
    signal:
        waveform-model: IMRPhenomXPHM
        minimum-frequency: 20
        detectors: [ET-2L-Aligned]
        output:
            output_directory: signal
            file_name: 'E-{{ detectors }}_BBH-{{ start_time }}-{{ duration }}.gwf'
            arguments: {channel: '{{ detectors }}:STRAIN'}
    noise:
        arguments: {psd_file: ET_10_full_cryo_psd, seed: 42}
        output:
            output_directory: noise
            file_name: 'E-{{ detectors }}_NOISE-{{ start_time }}-{{ duration }}.gwf'
            arguments: {channel: '{{ detectors }}:STRAIN'}

In [ ]:
!gwmock simulate config.yaml --author 'Workshop' --email 'you@example.org' 2>&1 | tail -2

## 2. Combine signal + noise with `gwmock merge`

`merge` **sums** all input frames that share the requested channel onto a
common time axis. Point it at a detector's signal and noise frame and you
get the analysis-ready strain. `--force` skips the (optional) metadata
bookkeeping for this quick demo.

In [ ]:
!gwmock merge \
  output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf \
  output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf \
  --channel ET1_2L_ALIGNED_SARD:STRAIN --output strain_E1.gwf --force 2>&1 | tail -2

In [ ]:
import numpy as np
from gwpy.timeseries import TimeSeries
ch = 'ET1_2L_ALIGNED_SARD:STRAIN'
data = TimeSeries.read('strain_E1.gwf', channel=ch)
sig  = TimeSeries.read('output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf', channel=ch)
noi  = TimeSeries.read('output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf', channel=ch)
print('merged == signal + noise :', np.allclose(data.value, sig.value + noi.value))

## 3. Hand the strain to an analysis

From here it's standard GW tooling. Frames carry channel, sample rate and
GPS time, so a pipeline can discover everything it needs. Two common moves:

**(a) Estimate the amplitude spectral density (ASD)** — the noise floor a
search whitens against:

In [ ]:
import matplotlib.pyplot as plt
asd = data.asd(fftlength=8)
plt.loglog(asd.frequencies.value, asd.value)
plt.xlim(5, 1024); plt.xlabel('frequency [Hz]'); plt.ylabel(r'ASD [strain/$\sqrt{Hz}$]')
plt.title('Merged strain ASD (ET1, 2L-aligned)'); plt.grid(True, which='both', alpha=0.3); plt.show()

**(b) Slice analysis segments** the way a pipeline would, keeping GPS time:

In [ ]:
seg = data.crop(data.t0.value + 96, data.t0.value + 112)  # a 16 s analysis chunk
print('segment GPS %.0f-%.0f, %d samples @ %s'
      % (seg.t0.value, seg.t0.value + seg.duration.value, seg.size, seg.sample_rate))

## 4. Scaling up: `gwmock batch`

Real MDCs are many segments. `gwmock batch` turns a config into a Slurm
submit script (it does **not** submit unless you pass `--submit`):

```bash
gwmock batch config.yaml --job-name mdc --account my_alloc --time 04:00:00
```

Each segment is independently seeded (notebook 02), so array jobs are
reproducible and resumable.

## 5. Sharing: `gwmock repository`

`gwmock repository create|upload|download|verify` moves a dataset to/from a
**Zenodo** deposition, carrying the metadata so downloaders can `validate`
what they pulled.

➡️ Final: `04-advanced-sgwb.ipynb`.